# Huấn luyện YOLOv8n với BDD100K cho ADAS

Chạy trên Google Colab: Runtime → Change runtime type → GPU, rồi chạy lần lượt các ô.

Mô hình học từ BDD100K rồi nhận diện video mới; không cần học từng video. Sáu nhãn: person, bicycle, motorcycle, car, bus, truck. pedestrian và rider được gộp thành person. Đây là mô hình ảnh 2D, không đo khoảng cách LiDAR.

Nguồn: [BDD100K download](https://github.com/bdd100k/bdd100k/blob/master/doc/source/download.rst), [định dạng nhãn](https://github.com/bdd100k/bdd100k/blob/master/doc/source/format.rst), [YOLO training](https://docs.ultralytics.com/modes/train/). Xem giấy phép BDD100K trước khi sử dụng.

Tải **Images 100K** và **Detection 2020 Labels** từ nguồn chính thức. Không cần Videos 1.8 TB. Đặt ZIP/TAR vào MyDrive/ADAS_BDD100K/input; notebook sẽ giải nén trong ổ tạm Colab. Không tải hàng nghìn ảnh riêng lên Drive.

In [ ]:
%pip install -q ultralytics==8.3.228 pillow pyyaml requests
from pathlib import Path
import torch, json, random, shutil, os
from google.colab import drive
drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Hãy chọn GPU trong Runtime > Change runtime type'
print(torch.cuda.get_device_name(0))
BASE = Path('/content/drive/MyDrive/ADAS_BDD100K')
INPUT = BASE / 'input'
INPUT.mkdir(parents=True, exist_ok=True)
print('Đặt các gói dữ liệu tại:', INPUT)
print('Ổ Colab trống (GB):', round(shutil.disk_usage('/content').free / 2**30, 1))

## 1. Cấu hình

Lần đầu giữ QUICK=True và EPOCHS=3 để kiểm tra. Khi đã chạy ổn, đổi QUICK=False, EPOCHS=50, RUN_NAME='bdd_full_v1'. Không dùng chung tên thư mục cho các thử nghiệm khác nhau.

Nếu có URL tải trực tiếp chính thức, điền ARCHIVE_URLS; nếu không, tải bằng trình duyệt và đưa các gói nén vào thư mục input trên Drive. URL phải trả về ZIP/TAR, không phải trang web.

In [ ]:
QUICK = True
EPOCHS = 3
RUN_NAME = 'bdd_quick_v1'
RESUME = False  # True khi tiếp tục đúng thử nghiệm này sau khi Colab ngắt
BATCH = 8  # giảm xuống 4 nếu thiếu bộ nhớ GPU
ARCHIVE_URLS = {
    # 'images.zip': 'URL trực tiếp của Images 100K',
    # 'det_labels.zip': 'URL trực tiếp của Detection 2020 Labels',
}
RAW = Path('/content/bdd_raw')
DATA = Path('/content/bdd_yolo_quick' if QUICK else '/content/bdd_yolo_full')
RUN_DIR = BASE / 'runs' / RUN_NAME
if not RESUME and RUN_DIR.exists():
    raise RuntimeError('RUN_NAME đã tồn tại. Đổi tên hoặc bật RESUME để tiếp tục.')
RAW.mkdir(exist_ok=True)
print('Chế độ:', 'thử nhanh' if QUICK else 'toàn bộ', 'Kết quả:', RUN_DIR)

In [ ]:
import requests, zipfile, tarfile
for filename, url in ARCHIVE_URLS.items():
    assert Path(filename).name == filename
    target = INPUT / filename
    if target.exists():
        print('Đã có:', target.name)
        continue
    partial = target.with_name(target.name + '.part')
    with requests.get(url, stream=True, timeout=(30, 120)) as response:
        response.raise_for_status()
        with partial.open('wb') as out:
            for chunk in response.iter_content(8 * 1024 * 1024):
                out.write(chunk)
    if not (zipfile.is_zipfile(partial) or tarfile.is_tarfile(partial)):
        raise ValueError(f'{filename} không phải ZIP/TAR; kiểm tra URL tải')
    partial.replace(target)

archives = sorted(p for p in INPUT.iterdir() if p.is_file() and not p.name.endswith('.part'))
assert archives, f'Chưa có gói dữ liệu. Tải Images 100K và Detection 2020 Labels vào {INPUT}'
for archive in archives:
    marker = RAW / (archive.name + '.extracted')
    if marker.exists():
        continue
    if zipfile.is_zipfile(archive):
        with zipfile.ZipFile(archive) as z:
            for name in z.namelist():
                if not (RAW / name).resolve().is_relative_to(RAW.resolve()):
                    raise ValueError('Đường dẫn không hợp lệ trong ZIP')
            z.extractall(RAW)
    elif tarfile.is_tarfile(archive):
        with tarfile.open(archive) as t:
            t.extractall(RAW, filter='data')
    else:
        print('Bỏ qua tệp không phải gói nén:', archive.name)
        continue
    marker.touch()
    print('Đã giải nén:', archive.name)

## 2. Chuyển nhãn sang YOLO

Giữ tập train/val riêng của BDD100K. Không dùng tập test thiếu nhãn để đánh giá. Chọn ngẫu nhiên cố định trong chế độ thử; số ảnh in ra có thể nhỏ hơn hạn mức nếu thiếu ảnh. Notebook báo lỗi nếu có ảnh bị thiếu thay vì âm thầm bỏ qua.

Không ghép trực tiếp với nhãn COCO vì thứ tự mã lớp khác nhau. Dự án đọc tên lớp từ mô hình nên giữ các tên tiếng Anh dưới đây.

In [ ]:
from PIL import Image
from collections import Counter
NAMES = ['person', 'bicycle', 'motorcycle', 'car', 'bus', 'truck']
CATEGORY = {name: i for i, name in enumerate(NAMES)}
CATEGORY.update({'pedestrian': 0, 'rider': 0, 'bike': 1, 'motor': 2})

def yolo_line(label, width, height):
    category = label.get('category')
    box = label.get('box2d')
    if category not in CATEGORY or not box:
        return None
    x1, x2 = sorted((float(box['x1']), float(box['x2'])))
    y1, y2 = sorted((float(box['y1']), float(box['y2'])))
    x1, x2 = max(0, min(width, x1)), max(0, min(width, x2))
    y1, y2 = max(0, min(height, y1)), max(0, min(height, y2))
    if x2 <= x1 or y2 <= y1:
        return None
    return f'{CATEGORY[category]} {(x1+x2)/(2*width):.6f} {(y1+y2)/(2*height):.6f} {(x2-x1)/width:.6f} {(y2-y1)/height:.6f}'

# Kiểm tra phép chuyển đổi trước khi xử lý dữ liệu thật.
assert yolo_line({'category':'car','box2d':{'x1':10,'y1':20,'x2':50,'y2':60}},100,100) == '3 0.300000 0.400000 0.400000 0.400000'
assert yolo_line({'category':'pedestrian','box2d':{'x1':0,'y1':0,'x2':10,'y2':10}},100,100).startswith('0 ')
assert yolo_line({'category':'train','box2d':{}},100,100) is None

image_dirs = {}
for split in ('train', 'val'):
    candidates = [p for p in RAW.rglob(split) if p.is_dir() and any(p.glob('*.jpg'))]
    assert len(candidates) == 1, f'Cần đúng một thư mục ảnh {split}, tìm được: {candidates}'
    image_dirs[split] = candidates[0]

# Hỗ trợ file danh sách Scalabel 2020; dừng rõ ràng nếu người dùng chọn sai gói nhãn.
label_files = {}
for split in ('train', 'val'):
    candidates = [p for p in RAW.rglob('*.json') if split in p.stem.lower() and ('det' in str(p).lower() or 'labels_images' in p.stem)]
    assert len(candidates) == 1, f'Cần đúng một JSON detection {split}, tìm được: {candidates}. Chỉ dùng gói Detection 2020 Labels.'
    label_files[split] = candidates[0]

import yaml
summary = {}
for split in ('train', 'val'):
    records = json.loads(label_files[split].read_text())
    if isinstance(records, dict):
        records = records.get('frames', records.get('data'))
    assert isinstance(records, list), 'Gói nhãn không phải danh sách Scalabel frame'
    assert all('name' in frame and 'labels' in frame for frame in records), 'Sai cấu trúc nhãn detection'
    records = sorted(records, key=lambda f: f['name'])
    if QUICK:
        records = random.Random(42).sample(records, min(len(records), 5000 if split == 'train' else 1000))
    image_out, label_out = DATA / 'images' / split, DATA / 'labels' / split
    image_out.mkdir(parents=True, exist_ok=True)
    label_out.mkdir(parents=True, exist_ok=True)
    counts = Counter()
    for frame in records:
        name = Path(frame['name']).name
        src = image_dirs[split] / name
        assert src.is_file(), f'Thiếu ảnh: {src}'
        with Image.open(src) as image:
            width, height = image.size
        lines = []
        for label in frame['labels'] or []:
            line = yolo_line(label, width, height)
            if line:
                lines.append(line)
                counts[NAMES[int(line.split()[0])]] += 1
        (label_out / Path(name).with_suffix('.txt')).write_text('\n'.join(lines))
        dst = image_out / name
        if not dst.exists():
            os.symlink(src, dst)
    (DATA / f'{split}.txt').write_text('\n'.join(str(image_out / Path(f['name']).name) for f in records) + '\n')
    assert sum(counts.values()) > 0, 'Không có hộp nhận diện sau chuyển đổi'
    summary[split] = {'images': len(records), 'boxes': dict(counts)}
(DATA / 'data.yaml').write_text(yaml.safe_dump({'path': str(DATA), 'train': 'train.txt', 'val': 'val.txt', 'names': NAMES}))
print(json.dumps(summary, indent=2))
BASE.mkdir(exist_ok=True)
(BASE / f'{RUN_NAME}_dataset_summary.json').write_text(json.dumps(summary, indent=2))

## 3. Xem ảnh và nhãn trước khi train

Kiểm tra khung bao đúng vật thể. Nếu khung sai, dừng và kiểm tra gói dữ liệu.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
paths = (DATA / 'val.txt').read_text().splitlines()
fig, axes = plt.subplots(2, 2, figsize=(16, 9))
for ax, filename in zip(axes.flat, random.Random(7).sample(paths, min(4, len(paths)))):
    image = Image.open(filename)
    w, h = image.size
    ax.imshow(image)
    label_file = DATA / 'labels' / 'val' / Path(filename).with_suffix('.txt').name
    for line in label_file.read_text().splitlines():
        cls, x, y, bw, bh = map(float, line.split())
        left, top = (x-bw/2)*w, (y-bh/2)*h
        ax.add_patch(patches.Rectangle((left, top), bw*w, bh*h, fill=False, edgecolor='lime', linewidth=1))
        ax.text(left, top, NAMES[int(cls)], color='yellow', fontsize=8)
    ax.axis('off')
plt.tight_layout()

## 4. Huấn luyện và lưu trên Drive

Kết quả gồm best.pt (mô hình tốt nhất), last.pt (tiếp tục học), biểu đồ và bảng kết quả. Tiến trình được lưu sau mỗi epoch, nên nếu phiên dừng giữa epoch sẽ mất phần chưa lưu của epoch đó.

Sau khi Colab ngắt: kết nối lại GPU, chạy lại các ô chuẩn bị dữ liệu với cùng QUICK và RUN_NAME, đặt RESUME=True. Không tăng EPOCHS khi resume; muốn học thêm sau khi hoàn tất thì tạo thử nghiệm mới từ best.pt.

In [ ]:
from ultralytics import YOLO
if RESUME:
    checkpoint = RUN_DIR / 'weights' / 'last.pt'
    assert checkpoint.is_file(), f'Không có checkpoint: {checkpoint}'
    model = YOLO(str(checkpoint))
    model.train(resume=True, data=str(DATA / 'data.yaml'), device=0)
else:
    model = YOLO('yolov8n.pt')
    model.train(
        data=str(DATA / 'data.yaml'), epochs=EPOCHS, imgsz=640, batch=BATCH,
        device=0, workers=2, cache=False, seed=42, patience=15,
        project=str(BASE / 'runs'), name=RUN_NAME, exist_ok=False,
        save=True, save_period=1, plots=True,
    )
BEST = RUN_DIR / 'weights' / 'best.pt'
assert BEST.is_file(), f'Chưa có best.pt tại {BEST}'
print('Mô hình:', BEST)

## 5. Đánh giá

Xem recall của từng lớp, đặc biệt person, motorcycle và bicycle; không chỉ xem điểm tổng. Kết quả này trên BDD100K, chưa chứng minh chất lượng trên góc camera xe ben ở Việt Nam.

In [ ]:
best_model = YOLO(str(BEST))
metrics = best_model.val(data=str(DATA / 'data.yaml'), imgsz=640, batch=BATCH, device=0,
                         project=str(BASE / 'evaluations'), name=RUN_NAME, exist_ok=True)
print(metrics.results_dict)
from IPython.display import display
from IPython.display import Image as DisplayImage
if (RUN_DIR / 'results.png').exists():
    display(DisplayImage(filename=str(RUN_DIR / 'results.png')))

## 6. Nhận diện video bất kỳ (tùy chọn)

Đặt video trong Drive rồi sửa VIDEO. Video không cần gán nhãn hay học trước. Kết quả lưu trong Drive/ADAS_BDD100K/predictions. Chạy từng khung hình để không giữ toàn bộ video trong RAM.

In [ ]:
VIDEO = '/content/drive/MyDrive/ADAS_BDD100K/video_test.mp4'
assert Path(VIDEO).is_file(), f'Chưa tìm thấy video: {VIDEO}'
for result in best_model.predict(source=VIDEO, stream=True, save=True, conf=0.25,
                                 device=0, project=str(BASE / 'predictions'), name=RUN_NAME):
    pass
print('Video kết quả nằm tại:', BASE / 'predictions')

## 7. Tải mô hình về máy

Tải best.pt, đổi tên bdd_yolov8n.pt và đặt vào thư mục models của dự án. Khi tích hợp, đặt vision.model trong config/adas.yaml thành models/bdd_yolov8n.pt. Giữ allowed_classes gồm person, bicycle, motorcycle, car, bus, truck. Không thay cấu hình dự án trước khi đã kiểm tra mô hình trên video thực tế.

Chế độ thử chỉ để kiểm tra quy trình. Sau khi chạy ổn, dùng QUICK=False, EPOCHS=50 và RUN_NAME mới để huấn luyện chính thức. Thời gian tùy GPU Colab và dữ liệu; không bảo đảm hoàn tất trong một phiên.

In [ ]:
from google.colab import files
files.download(str(BEST))